# Binary Classification - Full Pipeline Persistence

The preprocessing and classifier are saved together. Inference receives raw features and automatically applies the same transformations used during training.

In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
)

In [ ]:
# ============================================================
# 1. EXAMPLE DATA
# ============================================================

rng = np.random.default_rng(42)

n = 2000

df = pd.DataFrame({
    "age": rng.integers(
        18,
        75,
        size=n
    ),
    "income": rng.normal(
        100000,
        30000,
        size=n
    ),
    "city": rng.choice(
        ["Colombo", "Kandy", "Galle"],
        size=n
    ),
    "job_level": rng.choice(
        ["junior", "mid", "senior"],
        size=n
    ),
})

score = (
    0.04 * (df["age"] - 40)
    + 0.00002 * (df["income"] - 100000)
    + (df["job_level"] == "senior") * 0.8
    + rng.normal(0, 1, n)
)

df["target"] = (
    score > 0.5
).astype(int)

print(df.head())
print(df["target"].value_counts(normalize=True))

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

TARGET = "target"

NUMERIC_COLUMNS = [
    "age",
    "income",
]

CATEGORICAL_COLUMNS = [
    "city",
    "job_level",
]

X = df.drop(columns=[TARGET])
y = df[TARGET]

In [ ]:
# ============================================================
# 3. SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

In [ ]:
# ============================================================
# 4. PREPROCESSING + MODEL PIPELINE
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        NUMERIC_COLUMNS
    ),
    (
        "categorical",
        categorical_pipeline,
        CATEGORICAL_COLUMNS
    ),
])

full_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )
    ),
])

In [ ]:
# ============================================================
# 5. TRAIN / EVALUATE
# ============================================================

full_pipeline.fit(
    X_train,
    y_train
)

predictions = full_pipeline.predict(
    X_test
)

probabilities = full_pipeline.predict_proba(
    X_test
)[:, 1]

print(
    classification_report(
        y_test,
        predictions
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        probabilities
    )
)

In [ ]:
# ============================================================
# 6. SAVE FULL PIPELINE
# ============================================================

MODEL_PATH = (
    "binary_classification_pipeline.joblib"
)

joblib.dump(
    full_pipeline,
    MODEL_PATH
)

print(
    "Saved:",
    MODEL_PATH
)

In [ ]:
# ============================================================
# 7. LOAD FULL PIPELINE
# ============================================================

loaded_pipeline = joblib.load(
    MODEL_PATH
)

print(
    type(loaded_pipeline)
)

In [ ]:
# ============================================================
# 8. INFERENCE USING RAW INPUT
# ============================================================

new_data = pd.DataFrame([
    {
        "age": 45,
        "income": 120000,
        "city": "Colombo",
        "job_level": "senior",
    }
])

prediction = loaded_pipeline.predict(
    new_data
)[0]

probability = loaded_pipeline.predict_proba(
    new_data
)[0, 1]

print(
    "Prediction:",
    int(prediction)
)

print(
    "Probability:",
    float(probability)
)

## Key Rule

Inference must use the same preprocessing learned during training.

Saving the full sklearn Pipeline keeps imputation, scaling, categorical encoding, and the classifier together. Raw production data can therefore be passed directly to `pipeline.predict(...)` without manually recreating transformations.